# Twitter Sentiment Analysis at Scale with PySpark

Distributed sentiment analysis of **~18.6 million tweets (~14 GB)** on an AWS EMR cluster.

The workflow: ingest four raw datasets from Amazon S3, clean and normalize the text with
distributed Spark transformations, generate sentiment labels with TextBlob, train a
TF-IDF + Naive Bayes classifier with Spark MLlib, and explore the results.

> This notebook was executed on an AWS EMR cluster (PySpark kernel via Livy) in April 2024.
> Cell outputs are preserved from that run. See the [README](../README.md) for the full
> write-up, results, and limitations.

## 1. Spark session

In [1]:
from pyspark.sql import SparkSession

Starting Spark application


SparkSession available as 'spark'.


In [2]:
spark = SparkSession.builder \
    .appName("Twitter Sentiment Analysis") \
    .getOrCreate()

## 2. Data ingestion from S3

The corpus is merged from four files (~14 GB in total): three CSV exports of tweets
mentioning the World Health Organization / COVID-19, and one JSON export of tweets from
US news accounts.

In [3]:
s3_path1 = 's3://tsa2/full_who_dataset1.csv'
s3_path2 = 's3://tsa/full_who_dataset2.csv'
s3_path3 = 's3://tsa/full_who_dataset3.csv'
s3_path4 = 's3://tsa2/us_news_tweets.json'

In [4]:
# The first three sources are CSV, the fourth is JSON
df1 = spark.read.csv(s3_path1, header=True, inferSchema=True)
df2 = spark.read.csv(s3_path2, header=True, inferSchema=True)
df3 = spark.read.csv(s3_path3, header=True, inferSchema=True)
df4 = spark.read.json(s3_path4)

### Schema inspection

Each source has a different schema. The only column needed for sentiment analysis is the
tweet text: `text` in the CSV sources, `content` in the JSON source.

In [5]:
df1.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- id_str: string (nullable = true)
 |-- from_user: string (nullable = true)
 |-- text: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- time: string (nullable = true)
 |-- geo_coordinates: string (nullable = true)
 |-- user_lang: string (nullable = true)
 |-- in_reply_to_user_id_str: string (nullable = true)
 |-- in_reply_to_screen_name: string (nullable = true)
 |-- from_user_id_str: string (nullable = true)
 |-- in_reply_to_status_id_str: string (nullable = true)
 |-- source: string (nullable = true)
 |-- profile_image_url: string (nullable = true)
 |-- user_followers_count: string (nullable = true)
 |-- user_friends_count: string (nullable = true)
 |-- user_location: string (nullable = true)
 |-- status_url: string (nullable = true)
 |-- entities_str: string (nullable = true)
 |--  : string (nullable = true)
 |-- tweet_lang: string (nullable = true)

In [6]:
df2.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- index: string (nullable = true)
 |-- id_str: string (nullable = true)
 |-- from_user: string (nullable = true)
 |-- text: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- time: string (nullable = true)
 |-- geo_coordinates: string (nullable = true)
 |-- user_lang: string (nullable = true)
 |-- in_reply_to_user_id_str: string (nullable = true)
 |-- in_reply_to_screen_name: string (nullable = true)
 |-- from_user_id_str: string (nullable = true)
 |-- in_reply_to_status_id_str: string (nullable = true)
 |-- source: string (nullable = true)
 |-- profile_image_url: string (nullable = true)
 |-- user_followers_count: string (nullable = true)
 |-- user_friends_count: string (nullable = true)
 |-- user_location: string (nullable = true)
 |-- status_url: string (nullable = true)
 |-- entities_str: string (nullable = true)
 |-- tweet_lang: string (nullable = true)

In [7]:
df3.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- level_0: string (nullable = true)
 |-- index: string (nullable = true)
 |-- id_str: string (nullable = true)
 |-- from_user: string (nullable = true)
 |-- text: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- time: string (nullable = true)
 |-- geo_coordinates: string (nullable = true)
 |-- user_lang: string (nullable = true)
 |-- in_reply_to_user_id_str: string (nullable = true)
 |-- in_reply_to_screen_name: string (nullable = true)
 |-- from_user_id_str: string (nullable = true)
 |-- in_reply_to_status_id_str: string (nullable = true)
 |-- source: string (nullable = true)
 |-- profile_image_url: string (nullable = true)
 |-- user_followers_count: string (nullable = true)
 |-- user_friends_count: string (nullable = true)
 |-- user_location: string (nullable = true)
 |-- status_url: string (nullable = true)
 |-- entities_str: string (nullable = true)
 |-- tweet_lang: string (nullable = true)

In [8]:
df4.printSchema()

root
 |-- _type: string (nullable = true)
 |-- cashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- content: string (nullable = true)
 |-- conversationId: long (nullable = true)
 |-- coordinates: struct (nullable = true)
 |    |-- _type: string (nullable = true)
 |    |-- latitude: double (nullable = true)
 |    |-- longitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- hashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- id: long (nullable = true)
 |-- inReplyToTweetId: long (nullable = true)
 |-- inReplyToUser: struct (nullable = true)
 |    |-- _type: string (nullable = true)
 |    |-- created: string (nullable = true)
 |    |-- description: string (nullable = true)
 |    |-- descriptionUrls: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- indices: array (nullable = true)
 |    |    |    |    |-- element: long (containsNull = true)
 |    |    |  

In [9]:
# Keep only the tweet text from each source
df1_tweets = df1.select('text')
df2_tweets = df2.select('text')
df3_tweets = df3.select('text')
df4_tweets = df4.select('content')

In [10]:
# Merge the four sources into a single DataFrame
df_tweets = df1_tweets.union(df2_tweets).union(df3_tweets).union(df4_tweets)

In [11]:
print("total rows of tweets before processing: ",df_tweets.count())

total rows of tweets before processing:  18627474

## 3. Text cleaning

The merged corpus is noisy: duplicated rows within and across sources, URLs, retweet
prefixes, line breaks, punctuation, and stop words. Cleaning is a chain of narrow,
fully distributed transformations:

1. Drop null rows and exact duplicates
2. Strip URLs
3. Replace line breaks with spaces
4. Remove retweet prefixes (`RT @username:`) and de-duplicate again
5. Lowercase and remove non-alphanumeric characters
6. Remove stop words

In [12]:
# Drop rows with null text
df_tweets_without_null_values = df_tweets.na.drop()

In [13]:
# Remove exact duplicate rows
df_tweets_without_duplicates = df_tweets_without_null_values.dropDuplicates()

In [14]:
print("total rows of tweets after removing null and duplicates: ", df_tweets_without_duplicates.count())

total rows of tweets after removing null and duplicates:  6089506

Null and duplicate removal shrinks the corpus from 18.6M to **6.09M** tweets — roughly
two thirds of the raw rows were duplicates within and across the merged sources.

In [15]:
df_tweets_without_duplicates.show(10,truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|text                                                                                                                                                                                                                                                                   |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|This Sleep Habit Doubles the Risk of Early Death in Women, Says Study\n\n#bodyhealth #sleep #news #mentalhealth #wellness\nhttps://t.co/HGdcvI8Z4e                                                       

In [16]:
# Strip URLs
from pyspark.sql.functions import regexp_replace
df_tweets_without_URLs = df_tweets_without_duplicates.withColumn('text', regexp_replace('text', 'http\S+', ''))

In [17]:
# Replace line breaks with spaces
df_tweets_without_linebreak = df_tweets_without_URLs.withColumn('text', regexp_replace('text', '\\n', ' '))

In [18]:
# Strip the retweet prefix so retweets collapse onto the original tweet text
df_cleaned_retweets1 = df_tweets_without_linebreak.withColumn('text', regexp_replace('text', '^RT @\w+: ', ''))

In [19]:
# With the RT prefix gone, retweets are now exact duplicates of the original tweet — drop them
df_cleaned_retweets2 = df_cleaned_retweets1.dropDuplicates()

In [20]:
df_cleaned_retweets2.count()

5392617

The second de-duplication pass leaves **5.39M** unique tweets.

In [21]:
# Rename the column to something more descriptive
df_cleaned_retweets2 = df_cleaned_retweets2.withColumnRenamed('text', 'tweets')

In [22]:
from pyspark.sql.functions import lower

# Normalize case
df_lowercased = df_cleaned_retweets2.withColumn('tweets', lower('tweets'))

In [23]:
# Keep only letters, digits and whitespace
df_without_specialcharacters = df_lowercased.withColumn('tweets', regexp_replace('tweets', '[^a-zA-Z0-9\s]', ' '))

In [24]:
df_without_specialcharacters.show(10, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|tweets                                                                                                                                                                                                                                                                                 |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|the trump admin forbids ny to use its  javitzcenter 1 000 bed field hospital for  covid19 but would not forbid u s  manufacturers of precious n 95 masks 

### Stop-word removal

A small custom stop-word list is applied through a Python UDF. Spark's built-in
`StopWordsRemover` with its full English list would be the natural upgrade — noted as
future work in the README.

In [25]:
stop_words = ['a', 'an', 'the', 'and', 'or', 'but', 'if', 'is', 'it', 'of', 'on', 'in', 'to', 'for', 'with', 'as', 'at', 'by', 'from', 'that', 'this', 'these', 'those', 'then', 'there', 'when', 'where', 'how', 'why']

In [26]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

remove_stopwords_udf = udf(remove_stopwords, StringType())

In [27]:
df = df_without_specialcharacters.withColumn('filtered_text', remove_stopwords_udf(df_without_specialcharacters.tweets))

In [28]:
tweets_without_stopwords=df.select("filtered_text")

In [29]:
tweets_without_stopwords.show(10, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|filtered_text                                                                                                                                                                                                                                                        |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|europa debe estar la altura proteger su ciudadan con el euco de hoy se inicia una din mica de negociaci n para alcanzar un acuerdo desde espa seguiremos defendiendo una respuesta europea frente al covid19 co

In [30]:
# Alias used by the modeling stages below
tweets = tweets_without_stopwords

## 4. Sentiment labeling with TextBlob

The corpus has no ground-truth sentiment labels, so labels are generated with
[TextBlob](https://textblob.readthedocs.io/): each tweet receives a polarity score in
[-1, 1], which is then bucketed into Negative / Neutral / Positive. This is a form of
weak supervision — the Spark MLlib classifier in the next section learns to reproduce
these labels at scale.

In [31]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
from textblob import TextBlob


def get_sentiment(text):
    blob = TextBlob(text)
    return blob.sentiment.polarity

sentiment_udf = F.udf(get_sentiment, FloatType())

tweets = tweets.withColumn("sentiment", sentiment_udf("filtered_text"))


In [32]:
tweets.show(5)

+--------------------+---------+
|       filtered_text|sentiment|
+--------------------+---------+
|reporte de covid1...|      0.0|
|       happening now|      0.0|
|el psoe de andalu...|      0.0|
|i signed francesc...|      0.0|
|coronavirus messi...|     -0.1|
+--------------------+---------+
only showing top 5 rows

In [33]:
# Bucket polarity scores into three classes
from pyspark.sql import functions as F

def categorize_sentiment(sentiment):
    if sentiment > 0.5:
        return "Positive"
    elif sentiment < -0.5:
        return "Negative"
    else:
        return "Neutral"

categorized_sentiment_udf = F.udf(categorize_sentiment, StringType())

tweets = tweets.withColumn("categorized_sentiment", categorized_sentiment_udf("sentiment"))

In [34]:
tweets.show(10)

+--------------------+----------+---------------------+
|       filtered_text| sentiment|categorized_sentiment|
+--------------------+----------+---------------------+
|flash jean michel...|       0.1|              Neutral|
|rt unbiodiversity...|       0.0|              Neutral|
|kayburley juniord...|       0.0|              Neutral|
|lesechos intervie...|       0.0|              Neutral|
|sambit patra has ...|       0.0|              Neutral|
|you should wear y...|       0.0|              Neutral|
|         565835706 0|       0.0|              Neutral|
|so so many people...|     0.425|              Neutral|
|preventing covid1...|      0.56|             Positive|
|through days thec...|0.20833333|              Neutral|
+--------------------+----------+---------------------+
only showing top 10 rows

In [35]:
counts_df = tweets.groupBy("categorized_sentiment").count()

In [36]:
counts_df.show()

+---------------------+-------+
|categorized_sentiment|  count|
+---------------------+-------+
|              Neutral|5205810|
|             Positive| 142720|
|             Negative|  44087|
+---------------------+-------+

With ±0.5 thresholds most tweets land in **Neutral** (5.21M), against 142.7K Positive
and 44.1K Negative. Two factors drive this: the sources (WHO and US news accounts) are
largely factual, report-style tweets rather than opinion, and ±0.5 is a deliberately
strict cut-off — only strongly polar language escapes the Neutral bucket.

In [37]:
# Persist the labeled dataset back to S3
tweets.write.csv("s3://tsa2/output.csv")

## 5. Training a distributed classifier (TF-IDF + Naive Bayes)

TextBlob scoring runs as a Python UDF, the slowest part of the pipeline. Training a
native Spark MLlib model on the TextBlob labels yields a classifier that can score new
tweets entirely inside the JVM:

- **Features** — tokenized text → hashed term frequencies (1,000 features) → TF-IDF
- **Model** — multinomial Naive Bayes, a fast and strong baseline for sparse text features
- **Split** — 70/30 train/test

In [38]:
# Fresh reference for the ML stages
df = tweets

In [39]:
from pyspark.ml.feature import HashingTF, IDF, StringIndexer, Tokenizer
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [40]:
# Convert sentiment category to numerical format
indexer = StringIndexer(inputCol="categorized_sentiment", outputCol="label")
df = indexer.fit(df).transform(df)

# Tokenize filtered_text
tokenizer = Tokenizer(inputCol="filtered_text", outputCol="words")
df = tokenizer.transform(df)

# Apply TF-IDF to convert words into features
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=1000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(hashingTF.transform(df))
df = idfModel.transform(hashingTF.transform(df))

# Split dataset into training and testing sets
train, test = df.randomSplit([0.7, 0.3])

# Train Naive Bayes model
nb = NaiveBayes()
nb_model = nb.fit(train)

# Make predictions on test data
predictions = nb_model.transform(test)

# Evaluate model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Accuracy:", accuracy)


Accuracy: 0.7957720867922707

In [41]:
# Precision
precision = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})
print("Precision:", precision)

Precision: 0.9543540928504965

In [42]:
recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})
print("Recall:", recall)

Recall: 0.7957720867922707

In [43]:
f1_score = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})
print("F1-score:", f1_score)

F1-score: 0.8600921408598173

The classifier reaches **79.6% accuracy** and **0.86 weighted F1** on the held-out
split. An honest caveat: with 96.5% of tweets labeled Neutral, a majority-class baseline
would beat these numbers on raw accuracy — class imbalance dominates the metrics.
Per-class evaluation and rebalancing are the first improvements to make (see the
README's limitations section).

## 6. Exploratory analysis

In [44]:
tweets_df=tweets.withColumnRenamed('filtered_text', 'tweets').withColumnRenamed('categorized_sentiment', 'sentiment_category').withColumnRenamed('sentiment', 'sentiment_score')

In [45]:
tweets_df.printSchema()

root
 |-- tweets: string (nullable = true)
 |-- sentiment_score: float (nullable = true)
 |-- sentiment_category: string (nullable = true)

In [46]:
df = tweets_df

In [47]:
from pyspark.sql.functions import col, lower, explode, split, desc, length

In [48]:
# Top keywords across the corpus (words of 5+ characters)
top_keywords = df.select(explode(split(lower(col("tweets")), "\s+")).alias("word")) \
                 .filter((length(col("word")) >= 5) & (col("word") != "") & (col("word") != "rt")) \
                 .groupBy("word").count() \
                 .orderBy(desc("count")).limit(10)
top_keywords.show()

+------------+-------+
|        word|  count|
+------------+-------+
|     covid19|1848596|
| coronavirus| 312020|
|       covid| 218357|
|breakingnews| 199664|
|      health| 174966|
|       about| 172230|
|      people| 164739|
|       after| 162070|
|       today| 154742|
|     ukraine| 144455|
+------------+-------+

## 7. Summary

| Stage | Tweets |
| --- | ---: |
| Raw merged corpus | 18,627,474 |
| After null + duplicate removal | 6,089,506 |
| After full cleaning | 5,392,617 |

![Sentiment distribution](../assets/sentiment_distribution.png)

![Top keywords](../assets/top_keywords.png)

**Takeaways**

- COVID-19 vocabulary dominates the corpus, consistent with the WHO-centric sources.
- Sentiment skews heavily neutral, a product of news-style sources and strict ±0.5 polarity thresholds.
- A TF-IDF + Naive Bayes model reproduces the TextBlob labels with 0.86 weighted F1, moving scoring from a Python UDF into native Spark MLlib.

Full discussion, charts, and limitations: [README](../README.md).